In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import dice_ml
from skimage.metrics import structural_similarity as ssim

from utils import load_original_images, compute_percentage_change, compute_img_quality, \
    load_human_generated_counterfactuals, compute_area_of_change, compare_area_of_change
from dnn import train_dnn

In [ ]:
dnn, (X_train, y_train), (X_test, y_test) = train_dnn()

In [ ]:
cf_data = load_human_generated_counterfactuals()

orig_data = load_original_images()
def _get_orig_img(digit_label: int):
    for d in orig_data:
        if d["label"] == digit_label:
            return d["img"]

In [ ]:
def _get_all_orig_digits(label: int):
    test_idx = y_test == label
    return X_test[test_idx, :]

In [ ]:
def compute_metrics(x_orig: np.ndarray, x_cf: np.ndarray, target_label: int) -> dict:
    return {"amount_of_change": np.sum(np.abs(x_orig - x_cf)),
            "percentage_of_change": compute_percentage_change(x_orig, x_cf),
            "area_of_change": compute_area_of_change(x_orig, x_cf),
            "img_quality": compute_img_quality(x_cf, _get_all_orig_digits(target_label))}

In [ ]:
def plot_avg_area_of_change(agg_results, f_out=None):
    area_of_change = {}
    for i in range(10):
        area_of_change[i] = {}

        for j in range(10):
            if i == j:
                continue
            area_of_change[i][j] = np.mean(agg_results[i][j]["area_of_change"], axis=0)


    fig = plt.figure(layout='constrained')
    plt.ylabel("Source digit")
    plt.xlabel("Target digit")

    idx = 1
    for i in range(10):
        for j in range(10):
            ax = plt.subplot(10, 10, idx)
            idx += 1
            if i == 0:
                ax.set_title(f"{j}", fontsize=16)
            if j == 0:
                ax.set_ylabel(f"{i}", rotation='horizontal', ha='right', fontsize=16)

            plt.xticks([])
            plt.yticks([])
            if i != j:
                if not np.isscalar(area_of_change[i][j]):
                    plt.imshow(area_of_change[i][j].reshape((28, 28)))
                    plt.xticks([])
                    plt.yticks([])
                    continue
            plt.imshow(np.repeat([np.nan], 28*28).reshape((28, 28)))
            plt.xticks([])
            plt.yticks([])

    fig.supxlabel('Target digit')
    fig.supylabel('Source digit')

    if f_out is not None:
        plt.savefig(f_out, bbox_inches="tight")
    else:
        plt.show()

### Human-generated Counterfactuals

In [ ]:
len(cf_data)

In [ ]:
human_cfs_eval = []
for orig_digit, target_digit, img in cf_data:
    orig_img = _get_orig_img(orig_digit)

    human_cfs_eval.append({"orig_digit": orig_digit, "target_digit": target_digit,
                           "metrics": compute_metrics(orig_img, img, target_digit)})
    
with open("human_generated_cfs-eval.pickle", "wb") as f_out:
    pickle.dump(human_cfs_eval, f_out)

In [ ]:
with open("human_generated_cfs-eval.pickle", "rb") as f_in:
    human_cfs_eval = pickle.load(f_in)

In [ ]:
def compare_with_human_generated_cfs(x_orig, x_cf, y_cf):
    agreement_scores = []
    similarity_scores = []

    for orig_digit, target_digit, img in cf_data:
        orig_img = _get_orig_img(orig_digit)

        if target_digit == y_cf:
            agreement_scores.append(compare_area_of_change(compute_area_of_change(orig_img, img),
                                                           compute_area_of_change(x_orig, x_cf)))
            similarity_scores.append(ssim(img.reshape((28, 28)), x_cf.reshape((28, 28)),
                                          data_range=img.max() - img.min()))
            
    return np.max(agreement_scores), np.max(similarity_scores)

In [ ]:
def process_results(agg_results_dict: dict, metric_keys: list) -> dict:
    r = {}

    for m_k in metric_keys:
        r[m_k] = {"latex_table": "", "data": [[] for _ in range(10)], "data_std": [[] for _ in range(10)]}

    for i in range(10):
        for m_k in metric_keys:
            r[m_k]["latex_table"] += f"& {i} &"

        for j in range(10):
            if i == j:
                for m_k in metric_keys:
                    r[m_k]["latex_table"] += " $-$ &"
                    r[m_k]["data"][i].append(np.nan)
                    r[m_k]["data_std"][i].append(np.nan)
            else:
                for m_k in metric_keys:
                    r[m_k]["data"][i].append(np.mean(agg_results_dict[i][j][m_k]))
                    r[m_k]["data_std"][i].append(np.std(agg_results_dict[i][j][m_k]))
                    r[m_k]["latex_table"] += f" ${np.round(np.mean(agg_results_dict[i][j][m_k]), 2)} \pm {np.round(np.std(agg_results_dict[i][j][m_k]), 2)}$ &"

        for m_k in metric_keys:
            r[m_k]["latex_table"] = r[m_k]["latex_table"][:-1] + "\\\\ \n"

    for m_k in metric_keys:
        r[m_k]["data"] = np.array(r[m_k]["data"])
        r[m_k]["data_std"] = np.array(r[m_k]["data_std"])

    return r

In [ ]:
len(human_cfs_eval)

In [ ]:
agg_human_results = {}
all_metrics = ["amount_of_change", "img_quality", "percentage_of_change", "area_of_change"]

for i in range(10):
    d = {}
    for j in range(10):
        if i == j:
            continue

        d[j] = {k: [] for k in all_metrics}
        agg_human_results[i] = d

for data in human_cfs_eval:
    i, j = data["orig_digit"], data["target_digit"]
    metrics = data["metrics"]

    for k in all_metrics:
        agg_human_results[i][j][k].append(metrics[k])

In [ ]:
res = process_results(agg_human_results, ["percentage_of_change", "img_quality"])

metric = "percentage_of_change"
#metric = "img_quality"
print(res[metric]["latex_table"])

fig = plt.figure()
plt.imshow(res[metric]["data"], cmap="viridis", vmin=0, vmax=1)
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 1), cmap='viridis'),
             orientation='vertical')
plt.xticks(list(range(10)))
plt.yticks(list(range(10)))
plt.ylabel("Source digit")
plt.xlabel("Target digit")
plt.savefig(f"human-gernated_{metric}.pdf", bbox_inches="tight")

pd.DataFrame(res[metric]["data"]).to_csv(f"human-gernated_{metric}_mean.csv")
pd.DataFrame(res[metric]["data_std"]).to_csv(f"human-gernated_{metric}_std.csv")

In [ ]:
plot_avg_area_of_change(agg_human_results, f_out="human_avg_area_of_change.pdf")

### Algorithm-generated Counterfactuals

#### Counterfactuals guided by prototypes

In [ ]:
with open("protoguided_cfs-eval.pickle", "rb") as f_in:
    protoguided_cfs_eval = pickle.load(f_in)

In [ ]:
len(protoguided_cfs_eval)

In [ ]:
agg_proto_results = {}
all_metrics = ["amount_of_change", "img_quality", "percentage_of_change", "area_of_change", "human_agreement_score", "human_similarity_score"]

for i in range(10):
    d = {}
    for j in range(10):
        if i == j:
            continue

        d[j] = {k: [] for k in all_metrics}
        agg_proto_results[i] = d

for data in protoguided_cfs_eval:
    i, j = data["orig_digit"], data["target_digit"]
    metrics = data["metrics"]

    for k in all_metrics:
        agg_proto_results[i][j][k].append(metrics[k])

In [ ]:
res = process_results(agg_proto_results, ["percentage_of_change", "img_quality", "human_agreement_score", "human_similarity_score"])

metric = "percentage_of_change"
#metric = "human_agreement_score"
#metric = "human_similarity_score"
#metric = "img_quality"
print(res[metric]["latex_table"])

fig = plt.figure()
plt.imshow(res[metric]["data"], cmap="viridis", vmin=0, vmax=1)
#plt.colorbar()
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 1), cmap='viridis'),
             orientation='vertical')
plt.xticks(list(range(10)))
plt.yticks(list(range(10)))
plt.ylabel("Source digit")
plt.xlabel("Target digit")
#plt.show()
plt.savefig(f"proto_{metric}.pdf", bbox_inches="tight")

pd.DataFrame(res[metric]["data"]).to_csv(f"proto_{metric}_mean.csv")
pd.DataFrame(res[metric]["data_std"]).to_csv(f"proto_{metric}_std.csv")


In [ ]:
plot_avg_area_of_change(agg_proto_results, f_out="proto_avg_area_of_change.pdf")

#### DiCE

In [ ]:
with open("dice_cfs-eval.pickle", "rb") as f_in:
    dice_cfs_eval = pickle.load(f_in)

In [ ]:
len(dice_cfs_eval)

In [ ]:
agg_dice_results = {}
all_metrics = ["amount_of_change", "img_quality", "percentage_of_change", "area_of_change", "human_agreement_score", "human_similarity_score"]

for i in range(10):
    d = {}
    for j in range(10):
        if i == j:
            continue

        d[j] = {k: [] for k in all_metrics}
        agg_dice_results[i] = d

for data in dice_cfs_eval:
    i, j = data["orig_digit"], data["target_digit"]
    
    x_orig = _get_orig_img(i)
    xcf = data["xcf"]

    """
    print(i, j)
    plt.figure()
    plt.imshow(x_orig.reshape((28, 28)))
    plt.figure()
    plt.imshow(xcf.reshape((28, 28)))
    
    area_of_change = compute_area_of_change(x_orig, xcf)
    plt.figure()
    plt.imshow(area_of_change.reshape((28, 28)))
    break
    """

    metrics = data["metrics"]

    for k in all_metrics:
        agg_dice_results[i][j][k].append(metrics[k])

In [ ]:
res = process_results(agg_dice_results, ["percentage_of_change", "img_quality", "human_agreement_score", "human_similarity_score"])

#metric = "percentage_of_change"
#metric = "img_quality"
#metric = "human_agreement_score"
metric = "human_similarity_score"
print(res[metric]["latex_table"])

fig = plt.figure()
plt.imshow(res[metric]["data"], cmap="viridis", vmin=0, vmax=1)
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 1), cmap='viridis'),
             orientation='vertical')
plt.xticks(list(range(10)))
plt.yticks(list(range(10)))
plt.ylabel("Source digit")
plt.xlabel("Target digit")
plt.savefig(f"dice_{metric}.pdf", bbox_inches="tight")

pd.DataFrame(res[metric]["data"]).to_csv(f"dice_{metric}_mean.csv")
pd.DataFrame(res[metric]["data_std"]).to_csv(f"dice_{metric}_std.csv")

In [ ]:
plot_avg_area_of_change(agg_dice_results, f_out="dice_avg_area_of_change.pdf")

#### Normalizing Flows

In [ ]:
with open("nf_cfs-eval.pickle", "rb") as f_in:
    nf_cfs_eval = pickle.load(f_in)

In [ ]:
len(nf_cfs_eval)

In [ ]:
agg_nf_results = {}
all_metrics = ["amount_of_change", "img_quality", "percentage_of_change", "area_of_change", "human_agreement_score", "human_similarity_score"]

for i in range(10):
    d = {}
    for j in range(10):
        if i == j:
            continue

        d[j] = {k: [] for k in all_metrics}
        agg_nf_results[i] = d

for data in nf_cfs_eval:
    i, j = data["orig_digit"], data["target_digit"]
    metrics = data["metrics"]

    for k in all_metrics:
        agg_nf_results[i][j][k].append(metrics[k])

In [ ]:
res = process_results(agg_nf_results, ["percentage_of_change", "img_quality", "human_agreement_score", "human_similarity_score"])

metric = "percentage_of_change"
#metric = "human_agreement_score"
#metric = "human_similarity_score"
#metric = "img_quality"
print(res[metric]["latex_table"])

fig = plt.figure()
plt.imshow(res[metric]["data"], cmap="viridis", vmin=0, vmax=1)
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(0, 1), cmap='viridis'),
             orientation='vertical')
plt.xticks(list(range(10)))
plt.yticks(list(range(10)))
plt.ylabel("Source digit")
plt.xlabel("Target digit")
plt.savefig(f"nf_{metric}.pdf", bbox_inches="tight")

pd.DataFrame(res[metric]["data"]).to_csv(f"nf_{metric}_mean.csv")
pd.DataFrame(res[metric]["data_std"]).to_csv(f"nf_{metric}_std.csv")

In [ ]:
plot_avg_area_of_change(agg_nf_results, f_out="nf_avg_area_of_change.pdf")